# Implementación numérica del SSH de transmons acoplados con SQUIDs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt
from scipy.special import genlaguerre, gammaln
import itertools
import scipy.sparse.linalg as spla
import scienceplots
plt.style.use(['science', 'notebook'])
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh



In [5]:
np.linspace(-np.pi, np.pi, 4, endpoint = False)

array([-3.14159265, -1.57079633,  0.        ,  1.57079633])

In [6]:
def compute_position_exponential(N, u): 
  matrix = np.zeros(shape=(N, N), dtype=complex)
  for m in range(N):
    for n in range(N):
      if m >= n:
        matrix[m][n] = (
          np.exp(-u**2/2)
          * np.exp(0.5 * (gammaln(n+1) - gammaln(m+1)))
          * (1.0j*u)**(m-n)
          * genlaguerre(n, m-n)(u**2)
        )
      else:
        matrix[m][n] = (
          np.exp(-u**2/2)
          * np.exp(0.5 * (gammaln(m+1) - gammaln(n+1)))
          * (1.0j*u)**(n-m)
          * genlaguerre(m, n-m)(u**2)
        )

  return qt.Qobj(matrix)

Ahora debemos ser cuidadosos porque cuando construimos los valores de $k$, por lo general vienen de a pares $k$, $-k$. Sin embargo el modo $-\pi$ no viene con su par sino que el par correspondiente es el mismo (ya que cualquier estado que sea igual a otro módulo 2 $\pi$ es el mismo). Por tanto tenemos que construir el hamiltoniano separando los distintos $k$ de los casos $k = \pi$ y $k =0$.

In [7]:
"""Hamiltoniano de una cadena de transmon acoplados por SQUIDs
N: número de celdas unidad (# transmon = 2N)
fock_photon: dimension del espacio de hilbert de los fotones
fock_chain: dimension del espacio de Hilbert de cada uno de los transmon
"""
def embed(ops_at_indices: dict, n_total: int, fock_dim: int) -> qt.Qobj:
    """
    Embed a set of local operators into the full tensor product space.
    
    ops_at_indices: {index: qobj, ...}  — sites where non-identity ops go
    n_total: total number of sites (2*N for SSH, A and B interleaved)
    fock_dim: local Hilbert space dimension
    """
    tensor_aid = [qt.qeye(fock_dim)] * n_total
    for idx, op in ops_at_indices.items():
        tensor_aid[idx] = op
    return qt.tensor(tensor_aid)
            
  
# En esta version no consideramos el flujo de los SQUIDs sobre el resonador  
def ssh_squid_chain(N, fock_photon, fock_transmon, E_c, E_J, omega_c, gamma, Ec1, Ec2, E_J1, E_J2):
    omega_q = np.sqrt(8 * E_c * E_J) - E_c
    
    capacitive_factor = np.sqrt(E_J * (E_c)**3)**(0.25)/np.sqrt(2)
    gc1 = capacitive_factor / Ec1
    gc2 = capacitive_factor / Ec2
    
    squid_factor = 2*np.sqrt(2 * E_c / E_J)
    gj1 = squid_factor * E_J1
    gj2 = squid_factor * E_J2
    
    # Términos fotónicos
    sin_term = -0.5j * (compute_position_exponential(fock_photon, gamma) - compute_position_exponential(fock_photon, -gamma))
    epsilon = omega_q - (gj1 + gj2) * sin_term
    pairing_lambda = 0.5 * (gj1 + gj2) * sin_term
    
    k_values = np.linspace(-np.pi, np.pi, N, endpoint= False)
    t_k     = [(gc1 + gj1 * sin_term) + np.exp(-1.0j * k) * (gc2 + gj2 * sin_term) for k in k_values]
    delta_k = [(gc1 - gj1 * sin_term) + np.exp(-1.0j * k) * (gc2 - gj2 * sin_term) for k in k_values]
    
    eye_transmon = qt.tensor([qt.qeye(fock_transmon) for _ in range(2*N)])
    H_ph = qt.tensor(omega_c * qt.num(fock_photon) + N * (2*(E_J1 + E_J2) - 0.5 * (gj1 + gj2)) * sin_term, eye_transmon)
    
    n_transmon = qt.num(fock_transmon)
    b_transmon    = qt.destroy(fock_transmon)
    bdag_transmon = b_transmon.dag()
    
    
    def make_n_k():
        ops = []
        for j in range(N):
            ops.append(embed({2*j:   n_transmon}, 2*N, fock_transmon) + embed({2*j+1: n_transmon}, 2*N, fock_transmon))
        return np.array(ops)
    
    def make_self_pairing_k():
        ops = []
        for j in range(N):
            mj = (N - j) % N
            
            if mj == j:
                ops.append(embed({2*j:   bdag_transmon * bdag_transmon}, 2*N, fock_transmon)
                        + embed({2*j+1: bdag_transmon * bdag_transmon}, 2*N, fock_transmon))
            else:
                ops.append(embed({2*j:   bdag_transmon, 2*mj:   bdag_transmon}, 2*N, fock_transmon)
                        + embed({2*j+1: bdag_transmon, 2*mj+1: bdag_transmon}, 2*N, fock_transmon))
        return np.array(ops)
    
    def make_hopping_k():
        ops  = []
        for j in range(N):
            ops.append(embed({2*j: bdag_transmon, 2*j+1: b_transmon}, 2*N, fock_transmon))
        return np.array(ops)
    
    def make_intersite_pairing_k():
        ops  = []
        for j in range(N):
            mj = (N - j) % N
            ops.append(embed({2*j: bdag_transmon, 2*mj+1: bdag_transmon}, 2*N, fock_transmon))
        return np.array(ops)
    
    n_k = qt.tensor(epsilon, np.sum(make_n_k()))
    self_pairing_k = qt.tensor(pairing_lambda, np.sum(make_self_pairing_k()))
    hopping_k = np.sum([qt.tensor(t, hopping) for t, hopping in zip(t_k, make_hopping_k())])
    intersite_pairing_k = np.sum([qt.tensor(delta, pairing) for delta, pairing in zip(delta_k, make_intersite_pairing_k())])
    return n_k + self_pairing_k + self_pairing_k.dag() + H_ph + hopping_k + hopping_k.dag() - intersite_pairing_k - intersite_pairing_k.dag()

In [8]:
def linear_ssh_squid_chain(N, fock_photon, fock_transmon, E_c, E_J, omega_c, gamma, Ec1, Ec2, E_J1, E_J2):
    omega_q = np.sqrt(8 * E_c * E_J) - E_c
    
    capacitive_factor = np.sqrt(E_J * (E_c)**3)**(0.25)/np.sqrt(2)
    gc1 = capacitive_factor / Ec1
    gc2 = capacitive_factor / Ec2
    
    squid_factor = 2*np.sqrt(2 * E_c / E_J)
    gj1 = squid_factor * E_J1
    gj2 = squid_factor * E_J2
    
    # Términos fotónicos
    sin_term = gamma * (qt.destroy(fock_photon) + qt.destroy(fock_photon).dag())
    epsilon = omega_q - (gj1 + gj2) * sin_term
    pairing_lambda = 0.5 * (gj1 + gj2) * sin_term
    
    k_values = np.linspace(-np.pi, np.pi, N, endpoint= False)
    t_k     = [(gc1 + gj1 * sin_term) + np.exp(-1.0j * k) * (gc2 + gj2 * sin_term) for k in k_values]
    delta_k = [(gc1 - gj1 * sin_term) + np.exp(-1.0j * k) * (gc2 - gj2 * sin_term) for k in k_values]
    
    eye_transmon = qt.tensor([qt.qeye(fock_transmon) for _ in range(2*N)])
    H_ph = qt.tensor(omega_c * qt.num(fock_photon) + N * (2*(E_J1 + E_J2) - 0.5 * (gj1 + gj2)) * sin_term, eye_transmon)
    
    n_transmon = qt.num(fock_transmon)
    b_transmon    = qt.destroy(fock_transmon)
    bdag_transmon = b_transmon.dag()
    
    
    def make_n_k():
        ops = []
        for j in range(N):
            ops.append(embed({2*j:   n_transmon}, 2*N, fock_transmon) + embed({2*j+1: n_transmon}, 2*N, fock_transmon))
        return np.array(ops)
    
    def make_self_pairing_k():
        ops = []
        for j in range(N):
            mj = (N - j) % N
            
            if mj == j:
                ops.append(embed({2*j:   bdag_transmon * bdag_transmon}, 2*N, fock_transmon)
                        + embed({2*j+1: bdag_transmon * bdag_transmon}, 2*N, fock_transmon))
            else:
                ops.append(embed({2*j:   bdag_transmon, 2*mj:   bdag_transmon}, 2*N, fock_transmon)
                        + embed({2*j+1: bdag_transmon, 2*mj+1: bdag_transmon}, 2*N, fock_transmon))
        return np.array(ops)
    
    def make_hopping_k():
        ops  = []
        for j in range(N):
            ops.append(embed({2*j: bdag_transmon, 2*j+1: b_transmon}, 2*N, fock_transmon))
        return np.array(ops)
    
    def make_intersite_pairing_k():
        ops  = []
        for j in range(N):
            mj = (N - j) % N
            ops.append(embed({2*j: bdag_transmon, 2*mj+1: bdag_transmon}, 2*N, fock_transmon))
        return np.array(ops)
    
    n_k = qt.tensor(epsilon, np.sum(make_n_k()))
    self_pairing_k = qt.tensor(pairing_lambda, np.sum(make_self_pairing_k()))
    hopping_k = np.sum([qt.tensor(t, hopping) for t, hopping in zip(t_k, make_hopping_k())])
    intersite_pairing_k = np.sum([qt.tensor(delta, pairing) for delta, pairing in zip(delta_k, make_intersite_pairing_k())])
    return n_k + self_pairing_k + self_pairing_k.dag() + H_ph + hopping_k + hopping_k.dag() - intersite_pairing_k - intersite_pairing_k.dag()

In [9]:
N = 2
fock_photon = 3
fock_transmon = 7

E_c = 1.0
E_J = 100.0
omega_q = np.sqrt(8 * E_c * E_J) - E_c
omega_c = omega_q * 10.0 # fuera de resonancia

gamma = 1.0e-3 # un valor bastante optimista para gamma
Ec1 = E_c * 5.0
Ec2 = E_c * 5.0

E_J1 = 500.0
E_J2 = -500.0

In [10]:
# --- build Hamiltonian (your function unchanged) ---
H_qobj = ssh_squid_chain(
    N, fock_photon, fock_transmon,
    E_c, E_J, omega_c, gamma,
    Ec1, Ec2, E_J1, E_J2
)

# Convert to sparse (CSR)
H_sparse = H_qobj.data.tocsr()
dim = H_sparse.shape[0]

# --- define matrix-free operator ---
def matvec(psi):
    return H_sparse @ psi

H_linop = spla.LinearOperator(
    shape=(dim, dim),
    matvec=matvec,
    dtype=np.complex128
)

# --- compute lowest eigenvalues ---
n_eigs = 5  # number of eigenvalues you want

eigs, vecs = spla.eigsh(
    H_linop,
    k=n_eigs,
    which='SA'  # smallest algebraic
)

print("Eigenvalues:")
print(eigs)

AttributeError: 'qutip.core.data.dia.Dia' object has no attribute 'tocsr'